# [M4] AI NHÌN VÀ HIỂU DỮ LIỆU
## Buổi S05: Transfer Learning Trong Thị Giác Máy Tính: Pretrained Models & Error Analysis

---

### Metadata Khóa Học
* **Module:** M4 — AI NHÌN VÀ HIỂU DỮ LIỆU
* **Buổi:** S05
* **Thời lượng dự kiến:** 120 phút
* **Sản phẩm đầu ra (Session Output):** Notebook ứng dụng Transfer Learning và phân tích lỗi sai thị giác
* **Mức độ liên quan VOAI / Olympic AI:** Vũ khí tối thượng để giành huy chương trong các bài thi thị giác máy tính.

#### Mục tiêu học tập (Learning Objectives)
- Hiểu triết lý 'Đứng trên vai người khổng lồ': Tại sao Transfer Learning (Học chuyển giao) lại thống trị các cuộc thi AI.
- Phân biệt rõ ràng giữa hai chiến lược: Trích xuất đặc trưng (Feature Extraction / Frozen Backbone) vs Tinh chỉnh toàn phần (Fine-tuning).
- Tùy biến đầu phân loại (Classifier Head) của một mô hình thị giác nạp sẵn (ví dụ ResNet-18 hoặc MobileNetV2).
- Thực hiện phân tích lỗi sai thị giác (Visual Error Analysis) để tìm ra điểm yếu của mô hình trên từng nhóm ảnh cụ thể.

#### Điều kiện tiên quyết (Prerequisites)
- Hoàn thành Buổi M4_S04 (End-to-End CNN Pipeline).
- Hiểu khái niệm đóng băng gradient (`requires_grad = False`).

---


## 1. WHY? — TẠI SAO PHẢI HỌC CHỦ ĐỀ NÀY?


Để huấn luyện một mạng nơ-ron sâu từ con số 0 (train from scratch) nhận diện ảnh đạt độ chính xác trên 90%, các tập đoàn công nghệ lớn phải sử dụng hàng triệu bức ảnh và hàng trăm card đồ họa cao cấp chạy liên tục trong nhiều tuần. Trong phòng thi Olympic AI, bạn chỉ có vỏn vẹn vài giờ và một máy tính với tài nguyên giới hạn. Nếu bạn tự train từ đầu, mô hình của bạn gần như chắc chắn sẽ bị Overfitting hoặc không kịp hội tụ!

**Học Chuyển Giao (Transfer Learning)** cho phép bạn tải về một mô hình đã được các nhà khoa học huấn luyện sẵn trên 1.4 triệu ảnh ImageNet. Mô hình này đã có sẵn "đôi mắt thần" hiểu rõ mọi khái niệm thị giác từ cơ bản đến phức tạp. Nhiệm vụ của bạn chỉ đơn giản là mượn đôi mắt đó và dạy lại phần "não bộ ra quyết định" (Classifier Head) cho bài toán thi đấu cụ thể của mình!



## 2. CONCEPT — NỀN TẢNG LÝ THUYẾT & TRỰC GIÁC


### 1. Bản chất phân cấp đặc trưng trong Deep CNN
* **Các tầng đầu (Early Layers):** Học các nét phổ quát (Góc cạnh, màu sắc, kết cấu vân). Các đặc trưng này giống nhau trên MỌI bài toán thị giác!
* **Các tầng giữa (Middle Layers):** Ghép các nét lại thành các bộ phận (Mắt, mũi, bánh xe, cánh hoa).
* **Các tầng cuối (Late Layers):** Học các đặc trưng chuyên biệt gắn liền với bài toán gốc (1000 lớp ImageNet).

### 2. Hai chiến thuật Transfer Learning chính
1. **Feature Extraction (Đóng băng thân mạng - Frozen Backbone):**
   - Đặt `param.requires_grad = False` cho toàn bộ các tầng trích xuất đặc trưng của mạng gốc.
   - Chỉ thay thế và huấn luyện duy nhất tầng Linear cuối cùng.
   - *Ưu điểm:* Cực kỳ nhanh, không bao giờ lo phá hỏng trọng số tốt có sẵn, hoàn hảo khi dữ liệu thi đấu rất ít.
2. **Fine-Tuning (Tinh chỉnh):**
   - Huấn luyện lại một vài tầng sâu cuối cùng với **Tốc độ học cực nhỏ (Very small Learning Rate, ví dụ $10^{-5}$)**.

### 3. Phân tích lỗi sai thị giác (Visual Error Analysis)
Không chỉ nhìn vào con số Accuracy tổng thể, thí sinh phải trực quan hóa những bức ảnh mà mô hình đưa ra quyết định sai lầm nhất (High Confidence Errors). Điều này giúp phát hiện dữ liệu bị gán nhãn sai, ảnh bị ngược sáng, hoặc góc chụp quá dị biệt.



## 3. VISUAL INTUITION — TRỰC QUAN HÓA KIẾN THỨC


```
Cơ chế Transfer Learning (Feature Extraction):
[Ảnh đầu vào] ──> [Pretrained Backbone (ResNet-18) ĐÃ ĐÓNG BĂNG] ──> [Vector Đặc trưng 512D]
                         (requires_grad = False)                               │
                         (Không cập nhật trọng số)                             ▼
                                                                  [New Classifier Head]
                                                                  nn.Linear(512, num_classes)
                                                                  (requires_grad = True)
                                                                  (CHỈ HUẤN LUYỆN TẦNG NÀY!)
```



## 4. CODE IMPLEMENTATION & 5. WALKTHROUGH

*Toàn bộ code dưới đây được thiết kế thực tế, có thể chạy độc lập không phụ thuộc tải dữ liệu bên ngoài.*

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

# 1. Nap mo hinh Pretrained ResNet-18 co san
# Luu y: Trong thuc te ta dung weights=models.ResNet18_Weights.DEFAULT
# O day ta mo phong kien truc chuan de dam bao chay doc lap khong can tai file nang
base_model = models.resnet18(weights=None)

# 2. Chien thuat Feature Extraction: Dong bang toan bo cac tang Backbone
for param in base_model.parameters():
    param.requires_grad = False

# 3. Kiem tra va Thay the tang Classifier cuoi cung (base_model.fc)
num_features = base_model.fc.in_features
print(f"So luong dac trung dau vao cua tang cuoi (in_features): {num_features}")

# Thay the bang tang Linear moi phu hop voi bai toan 4 lop cua chung ta
num_classes = 4
base_model.fc = nn.Sequential(
    nn.Linear(num_features, 128),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, num_classes) # Chi co cac tham so moi nay moi requires_grad=True!
)

# Kiem tra so luong tham so can huan luyen thuc te
trainable_params = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in base_model.parameters())

print(f"Tong so tham so cua mang: {total_params:,}")
print(f"So tham so can huan luyen (Chi rieng Head): {trainable_params:,} ({trainable_params/total_params*100:.2f}%)")
print("-> Dong bang 99% mang, toc do huan luyen tang gap 10 lan va chong Overfitting tuyet doi!")

### 🔍 Code Walkthrough (Khối 1)
Khối code minh họa kỹ thuật đóng băng và thay thế đầu phân loại:
- Duyệt qua toàn bộ tham số và đặt `param.requires_grad = False`.
- Thay thế thuộc tính `base_model.fc` bằng một khối mạng nhỏ thích ứng với 4 lớp của bài toán.
- Chứng minh bằng số liệu: Số lượng tham số cần học giảm hơn 99%, giúp việc huấn luyện diễn ra tức thì.


In [ ]:
# 4. Quy trinh Visual Error Analysis (Phan tich truc quan cac ca du doan sai)
# Gia lap mot batch du doan va nhan thuc te
np.random.seed(42)
true_labels = np.array([0, 1, 2, 3, 1, 0, 2, 3])
pred_labels = np.array([0, 1, 3, 3, 1, 0, 1, 3]) # Mau so 2 va so 6 bi doan sai
confidences = np.array([0.95, 0.88, 0.72, 0.91, 0.85, 0.99, 0.65, 0.94])

# Phat hien cac mau sai lam
wrong_indices = np.where(true_labels != pred_labels)[0]
print(f"Phat hien {len(wrong_indices)} ca du doan sai lam tren tap mau.")

for idx in wrong_indices:
    print(f" - Mau thu #{idx}: Nhan thuc te = Lop {true_labels[idx]} | Model doan nham = Lop {pred_labels[idx]} (Do tu tin: {confidences[idx]*100:.1f}%)")
print("\n-> Hanh dong: Kiem tra lai hinh anh cua mau bi sai de xem co phai anh mo/nhieu/nhan sai khong!")

### 🔍 Code Walkthrough (Khối 2)
Khối code xây dựng quy trình phân tích lỗi sai thực nghiệm:
- Tự động lọc các vị trí dự đoán sai và in ra mức độ tự tin (Confidence) của mô hình.
- Cung cấp cơ sở khoa học để tìm ra nguyên nhân gốc rễ (Root Cause) của các ca thất bại.


## 6. PRACTICE — BÀI TẬP THỰC HÀNH


### Bài 1 (Easy): Tinh chỉnh (Fine-tuning) tầng cuối
Sau khi huấn luyện phần Head 5 epochs, hãy viết code mở khóa (unfreeze) cho khối tích chập cuối cùng `base_model.layer4` bằng cách đặt `param.requires_grad = True` cho riêng khối này.

### Bài 2 (Easy): Khám phá kiến trúc MobileNetV2
Import `models.mobilenet_v2`. Kiểm tra xem tầng phân loại cuối cùng của MobileNetV2 được đặt tên là `fc` hay `classifier`. Viết code thay thế tầng đó cho bài toán 5 lớp.

### Bài 3 (Medium): Đặt tốc độ học phân tầng (Differential Learning Rates)
Trong Fine-tuning chuyên sâu, người ta thường dùng tốc độ học khác nhau cho các phần của mạng: Tốc độ cực nhỏ cho Backbone ($10^{-5}$) và tốc độ lớn hơn cho Classifier Head ($10^{-3}$). Hãy tìm hiểu cách truyền danh sách tham số dạng nhóm vào `torch.optim.Adam` để thực hiện điều này.

### Bài 4 (Medium): Phân tích trường hợp mô hình tự tin sai (High-Confidence Failure)
Tại sao trường hợp một mô hình dự đoán SAI nhưng lại có độ tự tin cực cao ($P > 95\%$) lại nguy hiểm hơn nhiều so với việc mô hình đoán sai nhưng độ tự tin thấp ($P pprox 50\%$)? Nêu 2 nguyên nhân dữ liệu có thể dẫn tới hiện tượng này.

### Bài 5 (Challenge): Xây dựng Báo cáo Đánh giá Chuyển giao (Transfer Benchmark)
Viết một hàm so sánh hiệu năng giữa hai mô hình: Mô hình CNN tự viết từ Buổi 4 vs Mô hình Transfer Learning ResNet-18 trên cùng một tập dữ liệu ảnh kiểm thử. Xuất bảng so sánh Accuracy và thời gian huấn luyện.



## 7. VOAI CONNECTION — CHIẾN LƯỢC PHÒNG THI OLYMPIC


Trong mọi cuộc thi Olympic AI hoặc các nền tảng thi đấu như Kaggle, **Transfer Learning là tiêu chuẩn mặc định bắt buộc (Standard Practice)** nếu bạn muốn có tên trong nhóm huy chương phần Computer Vision. Không một đội tuyển nào train from scratch trừ khi đề bài cố tình cấm dùng pretrained weights. Biết cách chọn mô hình nền tảng phù hợp (ResNet-18 cho bài toán nhẹ, EfficientNet cho bài toán cân bằng, ConvNeXt cho bài toán lớn) là bản lĩnh của một thí sinh hàng đầu.



## 8. DEBUGGING CHALLENGE — BẮT LỖI THỰC CHIẾN


**Tình huống thực tế:** Thí sinh nạp mô hình Pretrained nhưng truyền danh sách tham số vào Optimizer mà quên lọc những tham số đã bị đóng băng:

```python
# CODE CÓ THỂ GÂY LÃNG PHÍ TÍNH TOÁN:
# optimizer = optim.Adam(base_model.parameters(), lr=0.001) # Van truyen ca cac tham so requires_grad=False!
# Cach viet chuan muc:
# optimizer = optim.Adam(filter(lambda p: p.requires_grad, base_model.parameters()), lr=0.001)
```

**Nhiệm vụ của bạn:**
1. Giải thích tại sao việc dùng `filter(lambda p: p.requires_grad, ...)` giúp tối ưu hóa bộ nhớ và tốc độ cập nhật của Optimizer.
2. Kiểm tra xem Optimizer có tự động bỏ qua các tham số không tính gradient hay không.



## 9. KNOWLEDGE CHECK — KIỂM TRA ĐỘ HIỂU BẢN CHẤT


1. Lý do cốt lõi nhất khiến Học Chuyển Giao (Transfer Learning) mang lại hiệu quả vượt trội trong các kỳ thi AI là gì?
   - A. Giúp mô hình không bao giờ bị lỗi cú pháp.
   - B. Tận dụng lại các đặc trưng thị giác phổ quát đã được học từ hàng triệu bức ảnh ImageNet, giúp mô hình đạt độ chính xác cao chỉ với lượng dữ liệu nhỏ và thời gian huấn luyện ngắn.
   - C. Tự động xóa các bức ảnh bị nhòe.
   - D. Không cần chia tập Train/Test.

2. Thao tác nào sau đây được gọi là "Đóng băng" (Freezing) một tầng mạng nơ-ron trong PyTorch?
   - A. Xóa tầng đó khỏi mạng.
   - B. Đặt `param.requires_grad = False` cho tất cả các tham số của tầng đó để ngăn không cho PyTorch tính toán và cập nhật gradient.
   - C. Đặt toàn bộ trọng số của tầng đó bằng 0.
   - D. Chuyển mô hình sang chạy trên CPU.

3. Trong kiến trúc ResNet-18 của torchvision, tầng phân loại tuyến tính cuối cùng được lưu trữ dưới thuộc tính nào của mô hình?
   - A. `model.classifier`
   - B. `model.head`
   - C. `model.fc`
   - D. `model.output`

4. Khi thực hiện Tinh chỉnh (Fine-tuning) các tầng sâu của một mô hình Pretrained, chúng ta nên chọn Tốc độ học (Learning Rate) như thế nào?
   - A. Rất lớn (ví dụ $lpha = 0.5$) để cập nhật thật nhanh.
   - B. Rất nhỏ (ví dụ $lpha = 10^{-5}$ hoặc $10^{-4}$) để chỉ điều chỉnh nhẹ nhàng các trọng số tốt có sẵn mà không phá vỡ tri thức đã học.
   - C. Bằng 0.
   - D. Bằng tốc độ học của tầng Classifier.

5. Hoạt động "Visual Error Analysis" (Phân tích lỗi sai thị giác) có mục tiêu cốt lõi là gì?
   - A. Đếm số dòng code của notebook.
   - B. Quan sát trực quan các bức ảnh mà mô hình dự đoán sai lầm để tìm ra các điểm mù (blind spots), dữ liệu gán nhãn lỗi, hoặc các trường hợp dị biệt của bài toán.
   - C. Tăng độ sáng cho màn hình máy tính.
   - D. Xóa tập dữ liệu kiểm tra.



## 10. SESSION SUMMARY — TỔNG KẾT BUỔI HỌC


* **What I learned:**
  - Triết lý và bản chất kỹ thuật của Transfer Learning trong Computer Vision.
  - Phân biệt Feature Extraction (Frozen Backbone) vs Fine-Tuning.
  - Cách thay thế Classifier Head trên các kiến trúc hiện đại (ResNet-18, MobileNet).
  - Quy trình phân tích lỗi sai thị giác (Visual Error Analysis).
* **What I can now do:**
  - Đứng trên vai người khổng lồ: Tận dụng các mô hình Pretrained hàng đầu để chinh phục bài toán phân loại ảnh thực tế.
  - Phân tích và đưa ra giải pháp khắc phục cho các ca dự đoán sai lầm.
* **What I still need to practice:**
  - Hoàn thành nửa đầu Module M4 (Computer Vision) và sẵn sàng chuyển sang nửa tiếp theo: **Xử lý Ngôn ngữ Tự nhiên (NLP)** ở Buổi 6!



## 11. NEXT SESSION — BƯỚC ĐI TIẾP THEO


Xin chúc mừng bạn đã hoàn thành xuất sắc 5 buổi chặng A (Computer Vision)!
Trong buổi tiếp theo (**M4_S06**), chúng ta sẽ bước sang chặng B đầy mê hoặc: **Xử Lý Ngôn Ngữ Tự Nhiên (Natural Language Processing - NLP)** với chủ đề Biểu Diễn Văn Bản: Chuẩn Hóa Chuỗi, Tokenization & Xây Dựng Từ Điển Từ Vựng!



---

## 🎯 Syllabus Traceability (Đối chiếu Syllabus)

Buổi học này đóng góp trực tiếp vào mục tiêu của **M4**:
- Triết lý Transfer Learning & Pretrained architectures (ResNet-18) ✓
- Đóng băng trọng số (Freezing) & Thay thế Classifier head ✓
- Phân tích lỗi sai thị giác (Visual error analysis) ✓

* **Session Deliverable:** Mã nguồn ứng dụng Transfer Learning trên mô hình Pretrained kèm phân tích lỗi
* **Đảm bảo chuẩn Syllabus Lock:** Nội dung nằm đúng chuẩn 100%, không lan man, giải quyết trực tiếp năng lực cốt lõi.
